In [1]:
# Run once. The embedding model (about 90 MB) downloads on first use.
!pip install -q langchain-text-splitters langchain-huggingface langchain-chroma sentence-transformers

In [2]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

sample_sop = """
# Standard Operating Procedure: Faulty Hardware Return & Replacement
Version: 2.1 | Applicable to: Regional Logistics & Hub Operations

1. Intake Protocol:
When hardware arrives from a regional hub, the system engineer must log the package count immediately.
All serial numbers must be scanned and cross-referenced with the dispatch manifest in the internal tracker within 24 hours.

2. Escalation & Stock Shortage:
If replacement stock is unavailable at the central hub, the engineer must not keep the ticket in pending status for more than 4 hours.
The engineer is authorized to execute a lateral stock transfer from the nearest secondary hub using express courier delivery to prevent operational downtime.

3. Maintenance Checklist:
Before dispatching any repaired or replacement device back to a hub, a standardized checklist verifying device power-on, network connectivity, and firmware version must be signed off.
"""

Path("sop_sample.txt").write_text(sample_sop.strip() + "\n")
print("Sample SOP file written successfully.")

Sample SOP file written successfully.


In [3]:
SOP_PATH = "sop_sample.txt"
text = Path(SOP_PATH).read_text()


def fixed_size_chunks(text, chunk_size=300, chunk_overlap=50):
    """Baseline: split by character count, like the first version."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.create_documents([text], metadatas=[{"source": SOP_PATH}])


def section_chunks(text):
    """Split on the numbered headings so each SOP clause stays in one chunk."""
    header, *sections = re.split(r"\n(?=\d+\. )", text.strip())
    header_lines = header.splitlines()
    title = header_lines[0].lstrip("# ").strip()
    version = header_lines[1].strip() if len(header_lines) > 1 else ""
    return [
        Document(
            page_content=sec.strip(),
            metadata={
                "source": SOP_PATH,
                "section": sec.split(":")[0].strip(),
                "doc_title": title,
                "version": version,
            },
        )
        for sec in sections
    ]


baseline_chunks = fixed_size_chunks(text)
chunks = section_chunks(text)

print(f"Fixed-size chunks: {len(baseline_chunks)}")
print(f"Section-based chunks: {len(chunks)}\n")
for c in chunks:
    print(f"--- {c.metadata['section']} ({len(c.page_content)} chars) ---")
    print(c.page_content, "\n")

Fixed-size chunks: 5
Section-based chunks: 3

--- 1. Intake Protocol (246 chars) ---
1. Intake Protocol:
When hardware arrives from a regional hub, the system engineer must log the package count immediately.
All serial numbers must be scanned and cross-referenced with the dispatch manifest in the internal tracker within 24 hours. 

--- 2. Escalation & Stock Shortage (324 chars) ---
2. Escalation & Stock Shortage:
If replacement stock is unavailable at the central hub, the engineer must not keep the ticket in pending status for more than 4 hours.
The engineer is authorized to execute a lateral stock transfer from the nearest secondary hub using express courier delivery to prevent operational downtime. 

--- 3. Maintenance Checklist (209 chars) ---
3. Maintenance Checklist:
Before dispatching any repaired or replacement device back to a hub, a standardized checklist verifying device power-on, network connectivity, and firmware version must be signed off. 



In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)


def build_index(docs, name):
    """Create a fresh cosine-distance Chroma collection (safe to re-run)."""
    meta = {"hnsw:space": "cosine"}
    Chroma(collection_name=name, embedding_function=embedding_model, collection_metadata=meta).delete_collection()
    return Chroma.from_documents(docs, embedding_model, collection_name=name, collection_metadata=meta)


baseline_db = build_index(baseline_chunks, "sop_baseline")
vector_db = build_index(chunks, "sop_sections")
print("Both indexes built.")

W1101 20:40:52.197000 5723 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Both indexes built.


In [5]:
test_set = [
    ("What should I do if replacement stock is missing at the central hub?", "lateral stock transfer"),
    ("How long can a ticket stay pending when stock is unavailable?", "4 hours"),
    ("Who is allowed to move stock in from another hub?", "lateral stock transfer"),
    ("How quickly must serial numbers be checked against the manifest?", "within 24 hours"),
    ("What do I log first when hardware arrives from a hub?", "package count"),
    ("What checks come before sending a device back to a hub?", "firmware version"),
    ("Which delivery option avoids operational downtime?", "express courier"),
    ("What has to be signed off before dispatch?", "checklist"),
]


def hit_rate(db, k):
    hits, misses = 0, []
    for question, phrase in test_set:
        docs = db.similarity_search(question, k=k)
        if any(phrase in d.page_content for d in docs):
            hits += 1
        else:
            misses.append(question)
    return hits, misses


print(f"{'Setup':<28}{'k=1':>8}{'k=2':>8}")
for label, db in [("Fixed-size chunks (300)", baseline_db), ("Section-based chunks", vector_db)]:
    row = []
    for k in (1, 2):
        hits, _ = hit_rate(db, k)
        row.append(f"{hits}/{len(test_set)}")
    print(f"{label:<28}{row[0]:>8}{row[1]:>8}")

print("\nQuestions the fixed-size index misses at k=2:")
_, missed = hit_rate(baseline_db, 2)
for q in missed or ["(none)"]:
    print(" -", q)

print("\nQuestions the section-based index misses at k=2:")
_, missed = hit_rate(vector_db, 2)
for q in missed or ["(none)"]:
    print(" -", q)

Setup                            k=1     k=2
Fixed-size chunks (300)          7/8     7/8
Section-based chunks             8/8     8/8

Questions the fixed-size index misses at k=2:
 - What should I do if replacement stock is missing at the central hub?

Questions the section-based index misses at k=2:
 - (none)


In [6]:
probe_questions = [
    ("in scope", "What should I do if replacement stock is missing at the central hub?"),
    ("in scope", "How fast must serial numbers be checked?"),
    ("unrelated", "What is the leave policy for employees?"),
    ("unrelated", "How do I reset my email password?"),
    ("near miss", "What is the warranty period for replaced devices?"),
    ("near miss", "Who pays for the express courier?"),
]

THRESHOLD = 0.6  # placeholder: set this from the numbers printed below, then re-run this cell

print(f"{'Type':<11}{'Best distance':>14}   Question")
for kind, q in probe_questions:
    _, score = vector_db.similarity_search_with_score(q, k=1)[0]
    verdict = "answer" if score <= THRESHOLD else "refuse"
    print(f"{kind:<11}{score:>14.3f}   {q}  -> {verdict}")

Type        Best distance   Question
in scope            0.322   What should I do if replacement stock is missing at the central hub?  -> answer
in scope            0.516   How fast must serial numbers be checked?  -> answer
unrelated           0.781   What is the leave policy for employees?  -> refuse
unrelated           0.978   How do I reset my email password?  -> refuse
near miss           0.678   What is the warranty period for replaced devices?  -> refuse
near miss           0.650   Who pays for the express courier?  -> refuse


In [7]:
PROMPT_TEMPLATE = """You are an intelligent Operations Support Assistant.
Answer the question strictly using the provided context. If the answer is not in the context, say "I cannot find this in the SOP."

Context:
{context}

Question:
{question}

Answer:"""

REFUSAL = "I cannot find this in the SOP."


def ask(query, k=2):
    """Return (prompt, refusal). Exactly one of them is None."""
    results = vector_db.similarity_search_with_score(query, k=k)
    kept = [(doc, score) for doc, score in results if score <= THRESHOLD]
    if not kept:
        return None, REFUSAL
    context = "\n---\n".join(f"[{doc.metadata['section']}]\n{doc.page_content}" for doc, _ in kept)
    return PROMPT_TEMPLATE.format(context=context, question=query), None


query = "What should I do if replacement stock is missing at the central hub?"
prompt, refusal = ask(query)
print("=== RETRIEVED, WITH DISTANCES ===")
for doc, score in vector_db.similarity_search_with_score(query, k=2):
    print(f"[{score:.4f}] {doc.metadata['section']}")
print("\n=== FINAL PROMPT ===")
print(prompt or refusal)

print("\n=== OUT-OF-SCOPE TEST ===")
_, refusal = ask("What is the leave policy for employees?")
print(refusal or "A chunk was close enough, so it was NOT refused. Check the threshold.")

=== RETRIEVED, WITH DISTANCES ===
[0.3219] 2. Escalation & Stock Shortage
[0.6264] 3. Maintenance Checklist

=== FINAL PROMPT ===
You are an intelligent Operations Support Assistant.
Answer the question strictly using the provided context. If the answer is not in the context, say "I cannot find this in the SOP."

Context:
[2. Escalation & Stock Shortage]
2. Escalation & Stock Shortage:
If replacement stock is unavailable at the central hub, the engineer must not keep the ticket in pending status for more than 4 hours.
The engineer is authorized to execute a lateral stock transfer from the nearest secondary hub using express courier delivery to prevent operational downtime.

Question:
What should I do if replacement stock is missing at the central hub?

Answer:

=== OUT-OF-SCOPE TEST ===
I cannot find this in the SOP.
